In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')

In [ ]:
from PitchModel.Stuff.DataPrep.DataPrep import DataPrep
from PitchModel.Constants import DATA_PREP_BINARY_ALL_FILE

data_prep = DataPrep.Load_From_File("../" + DATA_PREP_BINARY_ALL_FILE)

In [ ]:
from PitchModel.Constants import db
from PitchModel.DBTypes import *
from PitchModel.PitchDBTypes import *

cursor = db.cursor()
pitch = DB_PitchStatcast.Select_From_DB(
    cursor=cursor,
    conditional="WHERE GameId=? AND PitchId=?",
    values=(632285, 3)
)[0]

In [ ]:
import copy

x_range = [x / 10 for x in range(-30, 31)]
z_range = [z / 10 for z in range(5, 46)]

pitches = []
xs = []
zs = []
for x in x_range:
    for z in z_range:
        p = copy.deepcopy(pitch)
        p.pX = x
        p.pZ = z
        pitches.append(p)
        xs.append(x)
        zs.append(z)

In [ ]:
from PitchModel.Stuff.Model.PitchModel import PitchModel
from PitchModel.Constants import device
from PitchModel.Stuff.Model.PitchModel import *

args = DEFAULT_COMBINED_SWINGRESULTS_ARGS
network = PitchModel(args=args, data_prep=data_prep).to(device)

opva_list = network.GetPitchOutput(data_prep, "../Models/", pitches)

In [ ]:
runExpectancyMatrix = DB_RunExpectancyMatrix.Select_From_DB(
    cursor=cursor,
    conditional="WHERE Year=? AND CountBalls=? AND CountStrikes=?",
    values=(pitch.Year, pitch.CountBalls, pitch.CountStrike)
)

valueBall = [x.DeltaRuns for x in runExpectancyMatrix if x.Result == 4][0]
valueStrike = [x.DeltaRuns for x in runExpectancyMatrix if x.Result == 1][0]
valueFoul = [x.DeltaRuns for x in runExpectancyMatrix if x.Result == 3][0]
valueHBP = [x.DeltaRuns for x in runExpectancyMatrix if x.Result == 6][0]

print(valueBall, valueStrike, valueFoul, valueHBP)

In [ ]:
def GetOpvaValue(opva_list : list[DB_Output_PitchValueAggregation]) -> list[float]:
    values = []
    
    for opva in opva_list:
        pitchValue = 0

        pitchValue += opva.combinedBall * valueBall
        pitchValue += (opva.combinedCalledStrike + (opva.combinedSwing * opva.combinedWhiff)) * valueStrike
        pitchValue += (opva.combinedSwing * opva.combinedFoul) * valueFoul
        pitchValue += opva.combinedHBP * valueHBP
        pitchValue += opva.combinedSwing * opva.combinedInPlay * opva.combinedInPlayExpected
        values.append(pitchValue)
    
    return values
    

In [ ]:
pitch_values = GetOpvaValue(opva_list)

In [ ]:
opva_list[len(opva_list) // 2].combinedCalledStrike

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({'x': xs, 'y': zs, 'z': pitch_values})
pivot_table = df.pivot(index='y', columns='x', values='z')
plt.figure(figsize=(6, 8))
ax = sns.heatmap(
        pivot_table, 
        fmt='.0f',
        cmap='RdBu_r',
        vmin=-0.20,
        vmax=0.20,
        linecolor='none', 
        linewidths=0.5,
    )

ax.invert_yaxis()